In [1]:
# processing Morphologist morphometry info csv of Atril, Biosca and Cermoi
# read in the rest of the info and combine with morphometry info
# add additional 'SCA_side' column to combined_measures_allInfo
# add asymmetry index
# write the above data to csv file if needed
############################################################################################
# statistical tests for morphometry   
#   normality tests for all and by hemisphere
#   Anova and Kruskal-Wallis tests for morphometry
#   Kruskal-Wallis tests for asymmetry index (shapes in Isomap and UMAP plus morphometry)

In [2]:
import numpy as np
import pandas as pd

In [3]:
###############################  Composing measures combing Atril, Biosca and Cemoi  ##############################

In [4]:
###################################  Prepare Atril measures  ##################################
# Load distance measures
curRoot = 'C'  # 'C' or 'D'
curProject = 'Atril'
curRegionLeft = 'S.C._left' # 'S.C.sylvian._left'
curRegionRight = 'S.C._right' # 'S.C.sylvian._right'

path_left = rf'{curRoot}:\B_projWIP\proj_ataxia\measures\{curProject}\{curRegionLeft}.csv'
path_right = rf'{curRoot}:\B_projWIP\proj_ataxia\measures\{curProject}\{curRegionRight}.csv'
try:
    left = pd.read_csv(path_left, index_col=0, header=0, sep=';')
    right = pd.read_csv(path_right, index_col=0, header=0, sep=';')    
except FileNotFoundError as e:
    print(f"Error: {e}")

# adding side info to the index and remove the 'side' column
left.index = 'L' + left.index + '_M0'
right.index = 'flip-R' + right.index + '_M0'
del left['side']
del right['side']
Atril_measures = pd.concat([left,right],axis=0)

In [5]:
###################################  Prepare Biosca measures  ##################################
curProject = 'Biosca'
curRegionLeft = 'S.C._left' # 'S.C.sylvian._left'
curRegionRight = 'S.C._right' # 'S.C.sylvian._right'

path_left = rf'D:\B_projWIP\proj_ataxia\measures\{curProject}\{curRegionLeft}.csv'
path_right = rf'D:\B_projWIP\proj_ataxia\measures\{curProject}\{curRegionRight}.csv'
try:
    left = pd.read_csv(path_left, index_col=0, header=0, sep=';')
    right = pd.read_csv(path_right, index_col=0, header=0, sep=';')    
except FileNotFoundError as e:
    print(f"Error: {e}")

left.index = 'L' + left.index + '_E1'
right.index = 'flip-R' + right.index + '_E1'
del left['side']
del right['side']
Biosca_measures = pd.concat([left,right],axis=0)

Error: [Errno 2] No such file or directory: 'D:\\B_projWIP\\proj_ataxia\\measures\\Biosca\\S.C._left.csv'


KeyError: 'side'

In [ ]:
###################################  Prepare Cermoi measures  ##################################
curProject = 'Cermoi'
curRegionLeft = 'S.C._left' # 'S.C.sylvian._left'
curRegionRight = 'S.C._right' # 'S.C.sylvian._right'

path_left = rf'D:\B_projWIP\proj_ataxia\measures\{curProject}\{curRegionLeft}.csv'
path_right = rf'D:\B_projWIP\proj_ataxia\measures\{curProject}\{curRegionRight}.csv'
try:
    left = pd.read_csv(path_left, index_col=0, header=0, sep=';')
    right = pd.read_csv(path_right, index_col=0, header=0, sep=';')    
except FileNotFoundError as e:
    print(f"Error: {e}")

left.index = 'L' + left.index + '_V1'
right.index = 'flip-R' + right.index + '_V1'
del left['side']
del right['side']
Cermoi_measures = pd.concat([left,right],axis=0)

In [ ]:
# Combining the three bases
combined_measures = pd.concat([Atril_measures, Biosca_measures, Cermoi_measures], axis=0)

In [ ]:
combined_measures.columns

In [ ]:
##################################  Loading existing information  #################################
curRegion = 'CSSyl'
file_path = rf'D:\B_projWIP\proj_ataxia\Combined_Select_CSV_first_time_point\{curRegion}_atrilBioscaCermoi_combined.csv'
combined = pd.read_csv(file_path)
combined.index = combined['subjName']
combined.index = combined.index.astype(str)

In [ ]:
combined.columns

In [ ]:
##############################  combining the morphometry measures and all the rest  ################################
combined_measures_allInfo = pd.merge(combined_measures, combined, left_index=True, right_index=True)

In [ ]:
combined_measures_allInfo.columns

In [ ]:
# Add additional 'SCA_side' column to combined_measures_allInfo
combined_measures_allInfo['SCA_side'] = combined_measures_allInfo['SCA'].astype(str) + '_' + combined_measures_allInfo['side']

####################  Add asymmetry Index: Separating sides, calculate asymmetry index, then recombine  #####################
# combined_complete replacing combined_measures_allInfo
# Separating sides, setting subjName as index
combined_complete_L = combined_measures_allInfo[combined_measures_allInfo['side'] == 'L'].set_index('subjName')
combined_complete_R = combined_measures_allInfo[combined_measures_allInfo['side'] == 'R'].set_index('subjName')

# Changing indexes so that we can calculate the asy index
combined_complete_L.index = combined_complete_L.index.str.lstrip('L')
combined_complete_R.index = combined_complete_R.index.str.lstrip('flip-R')

# Calculating and adding asy indexes
combined_complete_L['iso1_asy'] = (combined_complete_L['iso1'] - combined_complete_R['iso1'])/(combined_complete_L['iso1'] + combined_complete_R['iso1'])
combined_complete_R['iso1_asy'] = (combined_complete_L['iso1'] - combined_complete_R['iso1'])/(combined_complete_L['iso1'] + combined_complete_R['iso1'])
combined_complete_L['iso2_asy'] = (combined_complete_L['iso2'] - combined_complete_R['iso2'])/(combined_complete_L['iso2'] + combined_complete_R['iso2'])
combined_complete_R['iso2_asy'] = (combined_complete_L['iso2'] - combined_complete_R['iso2'])/(combined_complete_L['iso2'] + combined_complete_R['iso2'])
combined_complete_L['iso3_asy'] = (combined_complete_L['iso3'] - combined_complete_R['iso3'])/(combined_complete_L['iso3'] + combined_complete_R['iso3'])
combined_complete_R['iso3_asy'] = (combined_complete_L['iso3'] - combined_complete_R['iso3'])/(combined_complete_L['iso3'] + combined_complete_R['iso3'])

combined_complete_L['UMAP1_U1_asy'] = (combined_complete_L['UMAP1_U1']-combined_complete_R['UMAP1_U1'])/(combined_complete_L['UMAP1_U1']+combined_complete_R['UMAP1_U1'])
combined_complete_R['UMAP1_U1_asy'] = (combined_complete_L['UMAP1_U1']-combined_complete_R['UMAP1_U1'])/(combined_complete_L['UMAP1_U1']+combined_complete_R['UMAP1_U1'])
combined_complete_L['UMAP1_U2_asy'] = (combined_complete_L['UMAP1_U2']-combined_complete_R['UMAP1_U2'])/(combined_complete_L['UMAP1_U2']+combined_complete_R['UMAP1_U2'])
combined_complete_R['UMAP1_U2_asy'] = (combined_complete_L['UMAP1_U2']-combined_complete_R['UMAP1_U2'])/(combined_complete_L['UMAP1_U2']+combined_complete_R['UMAP1_U2'])
combined_complete_L['UMAP1_U3_asy'] = (combined_complete_L['UMAP1_U3']-combined_complete_R['UMAP1_U3'])/(combined_complete_L['UMAP1_U3']+combined_complete_R['UMAP1_U3'])
combined_complete_R['UMAP1_U3_asy'] = (combined_complete_L['UMAP1_U3']-combined_complete_R['UMAP1_U3'])/(combined_complete_L['UMAP1_U3']+combined_complete_R['UMAP1_U3'])
combined_complete_L['UMAP2_U3_asy'] = (combined_complete_L['UMAP2_U3']-combined_complete_R['UMAP2_U3'])/(combined_complete_L['UMAP2_U3']+combined_complete_R['UMAP2_U3'])
combined_complete_R['UMAP2_U3_asy'] = (combined_complete_L['UMAP2_U3']-combined_complete_R['UMAP2_U3'])/(combined_complete_L['UMAP2_U3']+combined_complete_R['UMAP2_U3'])
combined_complete_L['UMAP1_U4_asy'] = (combined_complete_L['UMAP1_U4']-combined_complete_R['UMAP1_U4'])/(combined_complete_L['UMAP1_U4']+combined_complete_R['UMAP1_U4'])
combined_complete_R['UMAP1_U4_asy'] = (combined_complete_L['UMAP1_U4']-combined_complete_R['UMAP1_U4'])/(combined_complete_L['UMAP1_U4']+combined_complete_R['UMAP1_U4'])
combined_complete_L['UMAP2_U4_asy'] = (combined_complete_L['UMAP2_U4']-combined_complete_R['UMAP2_U4'])/(combined_complete_L['UMAP2_U4']+combined_complete_R['UMAP2_U4'])
combined_complete_R['UMAP2_U4_asy'] = (combined_complete_L['UMAP2_U4']-combined_complete_R['UMAP2_U4'])/(combined_complete_L['UMAP2_U4']+combined_complete_R['UMAP2_U4'])

combined_complete_L['surface_talairach_asy'] = (combined_complete_L['surface_talairach'] - combined_complete_R['surface_talairach'])/(combined_complete_L['surface_talairach'] + combined_complete_R['surface_talairach'])
combined_complete_R['surface_talairach_asy'] = (combined_complete_L['surface_talairach'] - combined_complete_R['surface_talairach'])/(combined_complete_L['surface_talairach'] + combined_complete_R['surface_talairach'])
combined_complete_L['meandepth_talairach_asy'] = (combined_complete_L['meandepth_talairach'] - combined_complete_R['meandepth_talairach'])/(combined_complete_L['meandepth_talairach'] + combined_complete_R['meandepth_talairach'])
combined_complete_R['meandepth_talairach_asy'] = (combined_complete_L['meandepth_talairach'] - combined_complete_R['meandepth_talairach'])/(combined_complete_L['meandepth_talairach'] + combined_complete_R['meandepth_talairach'])
combined_complete_L['hull_junction_length_talairach_asy'] = (combined_complete_L['hull_junction_length_talairach'] - combined_complete_R['hull_junction_length_talairach'])/(combined_complete_L['hull_junction_length_talairach'] + combined_complete_R['hull_junction_length_talairach'])
combined_complete_R['hull_junction_length_talairach_asy'] = (combined_complete_L['hull_junction_length_talairach'] - combined_complete_R['hull_junction_length_talairach'])/(combined_complete_L['hull_junction_length_talairach'] + combined_complete_R['hull_junction_length_talairach'])
combined_complete_L['GM_thickness_asy'] = (combined_complete_L['GM_thickness'] - combined_complete_R['GM_thickness'])/(combined_complete_L['GM_thickness'] + combined_complete_R['GM_thickness'])
combined_complete_R['GM_thickness_asy'] = (combined_complete_L['GM_thickness'] - combined_complete_R['GM_thickness'])/(combined_complete_L['GM_thickness'] + combined_complete_R['GM_thickness'])
combined_complete_L['opening_asy'] = (combined_complete_L['opening'] - combined_complete_R['opening'])/(combined_complete_L['opening'] + combined_complete_R['opening'])
combined_complete_R['opening_asy'] = (combined_complete_L['opening'] - combined_complete_R['opening'])/(combined_complete_L['opening'] + combined_complete_R['opening'])

# Changing back the indexes so that we can concatenate L and R
combined_complete_L.index = 'L' + combined_complete_L.index.astype(str)
combined_complete_R.index = 'flip-R' + combined_complete_R.index.astype(str)
combined_complete = pd.concat([combined_complete_L, combined_complete_R], axis=0)


In [ ]:
combined_complete.columns

In [ ]:
# Write the DataFrame to a CSV file if needed
curRegion = 'CSSyl_S.C.'
#file_path = rf'D:\B_projWIP\proj_ataxia\combined_Select_CSV\{curRegion}_atrilBioscaCermoi_combined.csv'
#print(file_path)
#combined_complete.to_csv(file_path, index=True)  # write only if not already done!

# Test read the CSV file back into a DataFrame
#df_loaded = pd.read_csv(file_path)

#print(len(df_loaded))
#print("Data types:\n", df_loaded.dtypes)
#print(df_loaded)

In [ ]:
############################################   Statistical tests, shape and morphometry   ############################################

In [ ]:
# replace combined_complete by combined, add combined_L, combined_R and combined_CAG for further stats
combined = combined_complete
combined_L = combined_complete[combined_complete.index.str.startswith('L')]
combined_R = combined_complete[combined_complete.index.str.startswith('flip')]
combined_CAG = combined_complete.dropna(subset=['CAG'])

In [ ]:
# Shapiro-Wilk test for normality, not normally distributed if p-val < 0.05
from scipy.stats import shapiro

print('Shapiro-Wilk test for normality: combined')

curMeasure = 'iso1'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'iso2'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'iso3'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U1'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U2'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U3'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP2_U3'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U4'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP2_U4'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')

#'surface_talairach', 'surface_native', 'maxdepth_talairach','maxdepth_native', 'meandepth_talairach', 'meandepth_native'
#'hull_junction_length_talairach', 'hull_junction_length_native','GM_thickness', 'opening' 
curMeasure = 'surface_talairach'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'surface_native'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'maxdepth_talairach'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'maxdepth_native'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'meandepth_talairach'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'meandepth_native'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'hull_junction_length_talairach'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'hull_junction_length_native'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'GM_thickness'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'opening'
stat, p_value = shapiro(combined[curMeasure])
print(f'{curMeasure} p-val: {p_value}')

#'iso1_asy', 'iso2_asy', 'iso3_asy', 'UMAP1_U1_asy', 'UMAP1_U2_asy', 'UMAP1_U3_asy','UMAP2_U3_asy', 'UMAP1_U4_asy', 'UMAP2_U4_asy', 
#'surface_talairach_asy','meandepth_talairach_asy', 'hull_junction_length_talairach_asy','GM_thickness_asy', 'opening_asy'
curMeasure = 'iso1_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'iso2_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'iso3_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U1_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U2_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U3_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP2_U3_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U4_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP2_U4_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'surface_talairach_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'meandepth_talairach_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'hull_junction_length_talairach_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'GM_thickness_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'opening_asy'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')

In [ ]:
# Normality test by hemisphere

print('Shapiro-Wilk test for normality: all combined')
print('Left Hemisphere')

curMeasure = 'iso1'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'iso2'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'iso3'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U1'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U2'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U3'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP2_U3'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U4'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP2_U4'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')

#'surface_talairach', 'surface_native', 'maxdepth_talairach','maxdepth_native', 'meandepth_talairach', 'meandepth_native'
#'hull_junction_length_talairach', 'hull_junction_length_native','GM_thickness', 'opening' 
curMeasure = 'surface_talairach'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'surface_native'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'maxdepth_talairach'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'maxdepth_native'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'meandepth_talairach'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'meandepth_native'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'hull_junction_length_talairach'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'hull_junction_length_native'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'GM_thickness'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'opening'
stat, p_value = shapiro(combined_L[curMeasure])
print(f'{curMeasure} p-val: {p_value}')

print('Right Hemisphere')

curMeasure = 'iso1'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'iso2'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'iso3'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U1'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U2'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U3'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP2_U3'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP1_U4'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'UMAP2_U4'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')

#'surface_talairach', 'surface_native', 'maxdepth_talairach','maxdepth_native', 'meandepth_talairach', 'meandepth_native'
#'hull_junction_length_talairach', 'hull_junction_length_native','GM_thickness', 'opening' 
curMeasure = 'surface_talairach'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'surface_native'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'maxdepth_talairach'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'maxdepth_native'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'meandepth_talairach'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'meandepth_native'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'hull_junction_length_talairach'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'hull_junction_length_native'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'GM_thickness'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
curMeasure = 'opening'
stat, p_value = shapiro(combined_R[curMeasure])
print(f'{curMeasure} p-val: {p_value}')

In [ ]:
# Perform ANOVA test, assume normality

#'surface_talairach', 'surface_native', 'maxdepth_talairach','maxdepth_native', 'meandepth_talairach', 'meandepth_native'
#'hull_junction_length_talairach', 'hull_junction_length_native','GM_thickness', 'opening' 

from scipy.stats import f_oneway
anova_result = f_oneway(*[combined['surface_talairach'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("surface_talairach ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['surface_native'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("surface_native ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['maxdepth_talairach'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("maxdepth_talairach ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['maxdepth_native'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("maxdepth_native ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['meandepth_talairach'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("meandepth_talairach ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['meandepth_native'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("meandepth_native ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['hull_junction_length_talairach'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("hull_junction_length_talairach ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['hull_junction_length_native'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("hull_junction_length_native ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['GM_thickness'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("GM_thickness ANOVA p-value:", anova_result.pvalue)
anova_result = f_oneway(*[combined['opening'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("opening ANOVA p-value:", anova_result.pvalue)

In [ ]:
# Preform Kruskal-Wallis test, NOT assuming normality
from scipy.stats import kruskal

# Perform Kruskal-Wallis test
kruskal_result = kruskal(*[combined['iso1'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("iso1 Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['iso2'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("iso2 Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['iso3'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("iso3 Kruskal-Wallis p-value:", kruskal_result.pvalue)

kruskal_result = kruskal(*[combined['UMAP1_U1'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("UMAP1_U1 Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['UMAP1_U2'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("UMAP1_U2 Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['UMAP1_U3'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("UMAP1_U3 Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['UMAP2_U3'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("UMAP2_U3 Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['UMAP1_U4'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("UMAP1_U4 Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['UMAP2_U4'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("UMAP2_U4 Kruskal-Wallis p-value:", kruskal_result.pvalue)

#'surface_talairach', 'surface_native', 'maxdepth_talairach','maxdepth_native', 'meandepth_talairach', 'meandepth_native'
#'hull_junction_length_talairach', 'hull_junction_length_native','GM_thickness', 'opening' 
kruskal_result = kruskal(*[combined['surface_talairach'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("surface_talairach Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['surface_native'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("surface_native Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['maxdepth_talairach'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("maxdepth_talairach Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['maxdepth_native'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("maxdepth_native Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['meandepth_talairach'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("meandepth_talairach Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['meandepth_native'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("meandepth_native Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['hull_junction_length_talairach'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("hull_junction_length_talairach Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['hull_junction_length_native'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("hull_junction_length_native Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['GM_thickness'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("GM_thickness Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined['opening'][combined['SCA'] == group] for group in combined['SCA'].unique()])
print("opening Kruskal-Wallis p-value:", kruskal_result.pvalue)

In [ ]:
############################################   Asymmetry analysis   #############################################

In [ ]:
#'iso1_asy', 'iso2_asy', 'iso3_asy', 'UMAP1_U1_asy', 'UMAP1_U2_asy', 'UMAP1_U3_asy','UMAP2_U3_asy', 'UMAP1_U4_asy', 'UMAP2_U4_asy', 
#'surface_talairach_asy','meandepth_talairach_asy', 'hull_junction_length_talairach_asy','GM_thickness_asy', 'opening_asy'

kruskal_result = kruskal(*[combined_L['iso1_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("iso1_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['iso2_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("iso2_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['iso3_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("iso3_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)

kruskal_result = kruskal(*[combined_L['UMAP1_U1_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("UMAP1_U1_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['UMAP1_U2_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("UMAP1_U2_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['UMAP1_U3_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("UMAP1_U3_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['UMAP2_U3_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("UMAP2_U3_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['UMAP1_U4_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("UMAP1_U4_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['UMAP2_U4_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("UMAP2_U4_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)

#'surface_talairach_asy','meandepth_talairach_asy', 'hull_junction_length_talairach_asy','GM_thickness_asy', 'opening_asy'

kruskal_result = kruskal(*[combined_L['surface_talairach_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("surface_talairach_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['meandepth_talairach_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("meandepth_talairach_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['hull_junction_length_talairach_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("hull_junction_length_talairach_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['GM_thickness_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("GM_thickness_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
kruskal_result = kruskal(*[combined_L['opening_asy'][combined_L['SCA'] == group] for group in combined_L['SCA'].unique()])
print("opening_asy Kruskal-Wallis p-value:", kruskal_result.pvalue)
